In [9]:
"""
ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
FINAL HARDENING PHASE - ALL CRITICAL FIXES APPLIED
"""

import os
import time
import json
import warnings
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from dotenv import load_dotenv
import joblib

warnings.filterwarnings('ignore')
load_dotenv()

# Create directories
for d in ['validation', 'backtest', 'models']:
    os.makedirs(d, exist_ok=True)

# ========== CONFIGURATION ==========
# Feature sets 
SHORT_FEATURES = [
    'exchange_flow_share', 'net_exchange_flow_ratio', 'whale_exchange_flow_ratio',
    'whale_exchange_asymmetry', 'vol_ratio', 'btc_ret_lag1',  
    'btc_ret_lag3', 'eth_btc_corr_30d', 'whale_volume_ratio_delta_3d',
    'exchange_volume_zscore'
]

LONG_FEATURES = [
    'btc_rsi', 'vol_ratio', 'whale_volume_ratio', 'eth_rsi',  
    'btc_ret_lag1', 'eth_burned_zscore_90d', 'eth_btc_corr_30d',
    'eth_ret_lag1', 'btc_ret_lag7', 'btc_vol30',
    'whale_volume_ratio_delta_1d', 'whale_volume_ratio_delta_3d',
    'exchange_flow_share', 'net_exchange_flow_ratio',
    'whale_exchange_flow_ratio', 'tx_per_active_zscore_90d'
]

# Trading parameters
SLIPPAGE = 0.0008
FEES = 0.0004

# Entry thresholds - FROZEN (can tune later)
LONG_ENTRY_THRESHOLD = 0.50  # Lower for LONG to allow confirmation rescue
SHORT_ENTRY_THRESHOLD = 0.55  # Higher for SHORT (requires stronger signal)

# ========== UNIFIED CONFIDENCE & POSITION SIZING ==========

def adjust_confidence_unified(prob, regime, direction=None, veto_score=0, row=None):
    """
    UNIFIED CONFIDENCE ADJUSTMENT with optional funding modifier
    """
    # Base confidence from model
    base_conf = float(prob)
    
    # Apply veto boost (same for both directions)
    veto_boost = np.tanh(veto_score / 3) * 0.15
    
    # Initial adjusted confidence
    adj_conf = np.clip(base_conf + veto_boost, 0, 1)
    
    # Apply confidence caps based on regime
    if regime == "R3":
        max_conf = 0.70
    elif regime == "R5":
        max_conf = 0.85
    elif regime in ["R1", "R2"]:
        max_conf = 0.75
    else:
        max_conf = 0.95
    
    adj_conf = min(adj_conf, max_conf)
    
    # ====== OPTIONAL: Apply funding modifier (only if row provided) ======
    funding_reasons = []
    if row is not None and direction is not None:
        # This is optional - only applies in generate_unified_signal
        adj_conf, funding_reasons = apply_funding_modifier(row, direction, adj_conf)
    
    return adj_conf, funding_reasons

def map_confidence_to_size_unified(conf, regime=None, direction=None):
    """
    UNIFIED POSITION SIZING with explicit regime-aware thresholds
    """
    # ✅ EXPLICIT asymmetric confidence floors
    if direction == "LONG":
        if conf < 0.50:  # Lower threshold for LONG
            return 0.0
    elif direction == "SHORT":
        if conf < 0.55:  # Higher threshold for SHORT
            return 0.0
    else:
        if conf < 0.55:  # Default
            return 0.0
    
    # Base sizing scale (same for both directions)
    if conf < 0.60: 
        base_size = 0.25
    elif conf < 0.65: 
        base_size = 0.50
    elif conf < 0.70: 
        base_size = 0.75
    elif conf < 0.75: 
        base_size = 1.00
    elif conf < 0.80: 
        base_size = 1.25
    else: 
        base_size = 1.50
    
    # ✅ Apply regime-specific caps (explicit)
    if regime in ["R3", "R5"]:
        # SHORT regimes: conservative
        base_size = min(base_size, 1.0)
    elif regime in ["R1", "R2"]:
        # LONG regimes: moderate
        if direction == "LONG":
            base_size = min(base_size, 1.25)
        else:
            base_size = min(base_size, 1.0)
    else:
        base_size = min(base_size, 1.0)
    
    return base_size

# ========== UTILITY FUNCTIONS ==========
def to_utc(ts):
    """Ensure timestamp is UTC"""
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def rolling_zscore_safe(series, window=90):
    """FIXED: Shift AFTER calculation to prevent leakage"""
    return ((series - series.rolling(window).mean()) / 
            series.rolling(window).std()).shift(1)

def rolling_feature_safe(series, window, func='mean'):
    """Safe rolling with shift"""
    if func == 'mean':
        return series.rolling(window).mean().shift(1)
    elif func == 'std':
        return series.rolling(window).std().shift(1)
    elif func == 'median':
        return series.rolling(window).median().shift(1)
    return series

# ========== PRICE NOT NEAR LOWS HELPER ==========
def price_not_near_lows(row, df, lookback=90, min_pct=0.25):
    """
    ✅ FIX 1: Require price to be above X percentile of recent range
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return False
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True  # Not enough data
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct >= min_pct

# ========== PRICE NOT NEAR HIGHS HELPER ==========
def price_not_near_highs(row, df, lookback=90, max_pct=0.75):
    """
    Protection for LONG entries against buying local tops
    Only applies to LONG positions
    """
    if pd.isna(row['eth_price']):
        return True
    
    idx = row.name
    start_idx = max(0, idx - lookback)
    recent_prices = df.iloc[start_idx:idx]['eth_price'].values
    
    if len(recent_prices) < 10:
        return True
    
    price_min = np.min(recent_prices)
    price_max = np.max(recent_prices)
    
    if price_max - price_min < 1e-9:
        return True
    
    pct = (row['eth_price'] - price_min) / (price_max - price_min)
    return pct <= max_pct

# ========== DATA LOADING FROM FILES ==========
def load_data_from_files():
    """
    Load data from files saved by data_loader.py
    Now includes funding data
    """
    print("📂 Loading data from saved files...")
    
    files_to_load = {
        'whale_data': 'data/whale_ml_ready.csv',
        'market_intent': 'data/market_intent_ml_ready.csv', 
        'btc_price': 'data/price_cache/btc.csv',
        'eth_price': 'data/price_cache/eth.csv',
        'funding_data': 'data/funding_rates_ml_ready.csv'
    }
    
    loaded_data = {}
    
    for name, filepath in files_to_load.items():
        if os.path.exists(filepath):
            try:
                if 'price' in name:
                    df = pd.read_csv(filepath, parse_dates=["date"])
                    df["date"] = df["date"].apply(to_utc)
                else:
                    df = pd.read_csv(filepath, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                
                loaded_data[name] = df
                print(f"✅ Loaded {name}: {len(df)} rows")
                
                # Special handling for funding data
                if name == 'funding_data' and not df.empty:
                    # Verify funding data covers whale data dates
                    whale_dates = loaded_data.get('whale_data', pd.DataFrame())
                    if not whale_dates.empty:
                        funding_dates = df['block_date']
                        whale_min = whale_dates['block_date'].min()
                        whale_max = whale_dates['block_date'].max()
                        
                        funding_coverage = (
                            funding_dates.min() <= whale_min and
                            funding_dates.max() >= whale_max
                        )
                        
                        if funding_coverage:
                            print(f"   ✅ Full coverage: {whale_min.date()} to {whale_max.date()}")
                        else:
                            print(f"   ⚠️ Partial coverage")
                            print(f"   Whale: {whale_min.date()} to {whale_max.date()}")
                            print(f"   Funding: {funding_dates.min().date()} to {funding_dates.max().date()}")
                
            except Exception as e:
                print(f"❌ Failed to load {name}: {e}")
                loaded_data[name] = pd.DataFrame()
        else:
            print(f"❌ {name} file not found: {filepath}")
            if name == 'funding_data':
                print(f"   ⚠️ Funding data missing - system will use zeros")
            loaded_data[name] = pd.DataFrame()
    
    # Check essential data
    essential_data = ['whale_data', 'market_intent', 'btc_price', 'eth_price']
    if all(len(loaded_data[d]) > 0 for d in essential_data):
        print(f"\n✅ Essential data loaded successfully")
    else:
        print(f"\n⚠️  Some essential data files are missing or empty")
        print(f"   Please run data_loader.py to fetch fresh data")
    
    return (
        loaded_data.get('whale_data', pd.DataFrame()),
        loaded_data.get('market_intent', pd.DataFrame()),
        loaded_data.get('btc_price', pd.DataFrame()),
        loaded_data.get('eth_price', pd.DataFrame()),
        loaded_data.get('funding_data', pd.DataFrame())
    )

# ========== FEATURE ENGINEERING ==========
def add_price_features(df, price_col, prefix):
    """Add technical features for a price series"""
    df = df.copy()
    
    # Log returns
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lagged returns (including lag 2 for LONG confirmation)
    for lag in [1, 2, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std().shift(1)
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std().shift(1)
    
    # RSI
    returns = df[f'{prefix}_log_return']
    gains = returns.where(returns > 0, 0).rolling(14).mean()
    losses = -returns.where(returns < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = (100 - (100 / (1 + gains / (losses + 1e-10)))).shift(1)
    
    return df

def engineer_features(df_whales, df_market_intent, df_btc, df_eth, df_funding=None):
    """Engineer all features with funding data from loader"""
    print("🔧 Engineering features...")
    
    # Merge price data
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    
    # Merge with whale data
    df = pd.merge(
        df_whales, 
        df_prices, 
        left_on='block_date', 
        right_on='date', 
        how='left'
    ).drop(columns=['date'])
    
    # Merge with market intent data
    df = pd.merge(
        df, 
        df_market_intent, 
        on='block_date', 
        how='left', 
        suffixes=('', '_intent')
    )
    
    # ========== MERGE FUNDING DATA ==========
    if df_funding is not None and not df_funding.empty:
        # Merge but keep as separate column
        df = pd.merge(
            df,
            df_funding[['block_date', 'eth_funding_rate_8h']],
            on='block_date',
            how='left'  # LEFT join - keep all dates even if no funding
        )
        print(f"✅ Merged funding data: {len(df_funding)} rows")
        
        # Check what percentage has funding data
        funding_present = df['eth_funding_rate_8h'].notna().sum()
        funding_pct = funding_present / len(df) * 100
        print(f"   Funding coverage: {funding_present}/{len(df)} rows ({funding_pct:.1f}%)")
    else:
        # Don't create the column if no data
        print("⚠️ No funding data - funding column will not be created")
    
    
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Add price features
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    # ETH/BTC ratio features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean().shift(1)
    df['eth_btc_corr_30d'] = df['eth_log_return'].shift(1).rolling(30) \
        .corr(df['btc_log_return'].shift(1)).shift(1)
        
    # ========== ADD VOL_RATIO FEATURE ==========
    if 'eth_vol7' in df.columns and 'eth_vol30' in df.columns:
        df['vol_ratio'] = df['eth_vol7'] / df['eth_vol30']
        df['vol_ratio'] = df['vol_ratio'].clip(0.5, 2.0)
        print("✅ Created vol_ratio feature")
    else:
        print("⚠️  Could not create vol_ratio - missing eth_vol7 or eth_vol30")
        df['vol_ratio'] = 1.0
            
    # Apply safe rolling z-scores
    zscore_pairs = [
        ('whale_tx_count', 'whale_tx_zscore_90d'),
        ('tx_per_active', 'tx_per_active_zscore_90d'),
        ('eth_burned', 'eth_burned_zscore_90d'),
        ('exchange_volume', 'exchange_volume_zscore'),
    ]
    
    for raw_col, zscore_col in zscore_pairs:
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore_safe(df[raw_col], 90)
    
    # Burn/issuance ratio
    if all(col in df.columns for col in ['eth_burned', 'total_gas_fees']):
        df['burn_issuance_ratio'] = (df['eth_burned'] / (df['total_gas_fees'] + 1e-10)).shift(1)
    
    # Whale volume deltas
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1).shift(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3).shift(1)
    
    # Ensure ALL LONG_FEATURES exist
    for feature in LONG_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing LONG feature: {feature}")
            df[feature] = 0.0
    
    # Ensure ALL SHORT_FEATURES exist  
    for feature in SHORT_FEATURES:
        if feature not in df.columns:
            print(f"⚠️  Creating missing SHORT feature: {feature}")
            df[feature] = 0.0
    
    # Clean up intermediate columns
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    # Save engineered features
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features engineered: {len(df.columns)} columns, {len(df)} rows")
    print(f"   LONG features available: {sum(1 for f in LONG_FEATURES if f in df.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features available: {sum(1 for f in SHORT_FEATURES if f in df.columns)}/{len(SHORT_FEATURES)}")
    
    return df

# ========== TARGET CREATION ==========
def create_targets_two_tier(df, k=1.5):
    """
    Create two-tier SHORT labels (crash + breakdown)
    """
    print("🎯 Creating two-tier targets...")
    
    df = df.sort_values('block_date').reset_index(drop=True).copy()
    
    # Calculate returns
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    df['rolling_vol_30'] = df['eth_log_return'].rolling(30, min_periods=10).std()
    
    # T+2 returns and threshold
    df['return_t2'] = df['eth_log_return'].rolling(2).sum().shift(-2)
    
    # Dynamic threshold using 65th percentile
    df['threshold_t2'] = df['rolling_vol_30'].rolling(60, min_periods=20).quantile(0.65)
    
    # Tier 1: Crash (hard down)
    hard_down = (
        (df['return_t2'] < -df['threshold_t2']) &
        (df['eth_vol7'] > df['eth_vol30']).fillna(False)
    )
    
    # Tier 2: Breakdown (pre-crash)
    exchange_flow_median = df['exchange_flow_share'].rolling(90, min_periods=30).median()
    
    soft_down = (
        (df['eth_ret_lag1'].fillna(0) < 0) &
        (df['btc_ret_lag1'].fillna(0) < 0) &
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Create targets
    df['target_t2'] = 0
    df.loc[df['return_t2'] > df['threshold_t2'], 'target_t2'] = 1  # UP
    df.loc[hard_down | soft_down, 'target_t2'] = -1  # DOWN (both tiers)
    
    # Create binary targets
    df['y_long_t2'] = (df['target_t2'] == 1).astype(int)
    df['y_short_t2'] = (df['target_t2'] == -1).astype(int)
    
    # Clean up
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    
    # Print distribution
    print("\n📊 Target Distribution (Two-Tier SHORT):")
    for state, label in [(-1, 'DOWN'), (0, 'FLAT'), (1, 'UP')]:
        count = (df['target_t2'] == state).sum()
        percentage = count / len(df) * 100
        print(f"  {label:5s}: {count:4d} ({percentage:5.1f}%)")
    
    hard_count = hard_down.sum()
    soft_count = soft_down.sum()
    total_down = (df['target_t2'] == -1).sum()
    
    print(f"\n  Tier 1 (crash):     {hard_count:4d}")
    print(f"  Tier 2 (breakdown): {soft_count:4d}")
    print(f"  Total DOWN:         {total_down:4d}")
    
    return df

# ========== REGIME DEFINITION ==========
def define_regimes_extended(df):
    """Define trading regimes including R5 distribution regime"""
    print("📈 Defining extended regimes...")
    
    if 'btc_ret_lag1' not in df.columns or 'eth_vol7' not in df.columns:
        df['regime_code'] = 'R0'
        return df
    
    # Standard regimes based on BTC trend and ETH volatility
    btc_trend_7d = df['btc_ret_lag1'].rolling(7).mean()
    df['btc_regime'] = pd.cut(
        btc_trend_7d, 
        bins=[-np.inf, -0.005, 0.005, np.inf], 
        labels=['DOWN', 'FLAT', 'UP']
    )
    
    vol_median = df['eth_vol7'].rolling(180, min_periods=60).median()
    df['vol_regime'] = (df['eth_vol7'] > vol_median).map({True: 'HIGH', False: 'LOW'})
    
    # Combine for standard regimes
    df['regime'] = df['btc_regime'].astype(str) + '_' + df['vol_regime'].astype(str)
    regime_map = {
        'UP_HIGH': 'R1',    # Bull high vol
        'UP_LOW': 'R2',     # Bull low vol
        'DOWN_HIGH': 'R3',  # Bear high vol
        'DOWN_LOW': 'R4',   # Bear low vol
    }
    df['regime_code'] = df['regime'].map(regime_map).fillna('R0')
    
    # R5: Whale distribution regime
    exchange_flow_median = df['exchange_flow_share'].rolling(60, min_periods=20).median()
    
    df['dist_regime'] = (
        (df['whale_volume_ratio_delta_3d'].fillna(0) > 0) &
        (df['exchange_flow_share'] > exchange_flow_median).fillna(False)
    )
    
    # Override with R5 where distribution regime is active
    df.loc[df['dist_regime'], 'regime_code'] = 'R5'
    
    # Print regime distribution
    print("\n📊 Extended Regime Distribution:")
    regime_stats = []
    for code in ['R1', 'R2', 'R3', 'R4', 'R5', 'R0']:
        count = (df['regime_code'] == code).sum()
        if len(df) > 0:
            pct = count / len(df) * 100
            icon = '🟢' if code == 'R1' else ('🔴' if code in ['R3', 'R5'] else '⚪')
            regime_stats.append(f"{icon} {code}: {count:4d} ({pct:5.1f}%)")
    
    # Print in two columns
    for i in range(0, len(regime_stats), 2):
        row = regime_stats[i:i+2]
        print("  " + " | ".join(row))
    
    return df

# ========== BUILD COMPLETE PIPELINE ==========
def build_pipeline_complete(df_features):
    """
    Create the complete pipeline dataset with features, targets, and regimes
    """
    print("\n" + "="*70)
    print("BUILDING COMPLETE PIPELINE DATASET")
    print("="*70)
    
    # Create targets
    df_with_targets = create_targets_two_tier(df_features)
    
    # Define regimes
    df_complete = define_regimes_extended(df_with_targets)
    
    # Ensure all required features exist
    for feature in LONG_FEATURES + SHORT_FEATURES:
        if feature not in df_complete.columns:
            df_complete[feature] = 0.0
    
    # Fill NaN values for features
    feature_cols = [col for col in df_complete.columns if col not in 
                   ['block_date', 'target_t2', 'y_long_t2', 'y_short_t2', 
                    'regime_code', 'btc_regime', 'vol_regime', 'regime', 'dist_regime']]
    
    df_complete[feature_cols] = df_complete[feature_cols].fillna(method='ffill').fillna(0)
    
    # Save complete pipeline
    df_complete.to_csv('data/pipeline_complete.csv', index=False)
    
    # Report statistics
    print(f"\n✅ Pipeline complete saved:")
    print(f"   Rows: {len(df_complete)}")
    print(f"   Columns: {len(df_complete.columns)}")
    print(f"   Date range: {df_complete['block_date'].min().date()} to {df_complete['block_date'].max().date()}")
    print(f"   File: data/pipeline_complete.csv")
    
    # Feature availability report
    print(f"\n📊 Feature Availability:")
    print(f"   LONG features: {sum(1 for f in LONG_FEATURES if f in df_complete.columns)}/{len(LONG_FEATURES)}")
    print(f"   SHORT features: {sum(1 for f in SHORT_FEATURES if f in df_complete.columns)}/{len(SHORT_FEATURES)}")
    
    # Check for missing features
    missing_long = [f for f in LONG_FEATURES if f not in df_complete.columns]
    missing_short = [f for f in SHORT_FEATURES if f not in df_complete.columns]
    
    if missing_long:
        print(f"   ⚠️  Missing LONG features: {missing_long}")
    if missing_short:
        print(f"   ⚠️  Missing SHORT features: {missing_short}")
    
    return df_complete

# ========== SHORT-SPECIFIC LOGIC ==========
def check_r3_short_allowed(row):
    """
    R3 short philosophy (early weakness only)
    """
    # Small red, not dump
    small_red = (-0.015 < row.get('eth_ret_lag1', 0) < 0)
    
    # BTC weakening
    btc_weak = (row.get('btc_ret_lag3', 0) < 0)
    
    # NOT vol expansion (early, not panic)
    vol_ratio = row.get('vol_ratio', 1)
    no_vol_expansion = (vol_ratio <= 1.0)
    
    # Whales increasing activity
    whale_activity = (row.get('whale_volume_ratio_delta_3d', 0) > 0)
    
    return small_red and btc_weak and no_vol_expansion and whale_activity

def calculate_short_veto_score(row):
    """Calculate veto scores for SHORT positions"""
    veto = 0
    reasons = []
    
    structural_score = 0
    flow_score = 0
    context_score = 0
    
    # Flow vetoes
    if row.get('net_exchange_flow_ratio', 0) < 0 and row.get('exchange_volume_zscore', 0) > 0:
        veto += 1
        flow_score += 1
        reasons.append('net_flow_negative_with_liquidity')
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto += 1
        flow_score += 1
        reasons.append('whale_to_exchange')
    
    # Structural vetoes
    if row.get('btc_ret_lag1', 0) < -0.02 and row.get('eth_ret_lag1', 0) < -0.01:
        veto += 2
        structural_score += 2
        reasons.append('btc_breakdown')
    
    # Structural vetoes - UPDATED vol expansion check
    if row.get('vol_ratio', 1) > 1.0:  # CHANGED: eth_vol7 > eth_vol30 → vol_ratio > 1.0
        veto += 2
        structural_score += 2
        reasons.append('vol_expansion')
    
    # Context vetoes - UPDATED low volatility check
    if row.get('vol_ratio', 1) < 0.7:  # CHANGED: eth_vol7 < eth_vol30*0.7 → vol_ratio < 0.7
        veto += 1
        context_score += 1
        reasons.append('low_volatility')
    
    return veto, structural_score, flow_score, context_score, reasons
        
# ========== FUNDING CONFIDENCE MODIFIER ==========
def apply_funding_modifier(row, direction, current_confidence):
    """
    Apply funding rate as OPTIONAL confidence modifier
    Only when funding data exists (post-2025)
    """
    # Get funding rate - returns None if column doesn't exist or is NaN
    if 'eth_funding_rate_8h' not in row.index:
        return current_confidence, []  # No funding column at all
    
    funding = row.get('eth_funding_rate_8h')
    
    # CRITICAL: If funding is None or NaN, do NOTHING
    if funding is None or pd.isna(funding):
        return current_confidence, []
    
    reasons = []
    funding_adjustment = 0.0
    
    # Asymmetric thresholds (same as before but as modifier, not veto)
    FUNDING_EUPHORIA_LEVEL = 0.0003  # 0.03% for crowded longs
    FUNDING_PANIC_LEVEL = -0.0002    # -0.02% for crowded shorts
    
    if direction == "LONG":
        if funding > FUNDING_EUPHORIA_LEVEL:
            # Crowded longs reduce confidence
            funding_adjustment = -0.03
            reasons.append(f"crowded_longs_funding_{funding*100:.3f}%")
    
    elif direction == "SHORT":
        if funding < FUNDING_PANIC_LEVEL:
            # Crowded shorts reduce confidence
            funding_adjustment = -0.03
            reasons.append(f"crowded_shorts_funding_{funding*100:.3f}%")
        elif funding > FUNDING_EUPHORIA_LEVEL:
            # High positive funding boosts SHORT confidence
            funding_adjustment = +0.03
            reasons.append(f"euphoric_funding_boost_{funding*100:.3f}%")
    
    # Apply adjustment with bounds
    adjusted_confidence = current_confidence + funding_adjustment
    adjusted_confidence = max(0.0, min(1.0, adjusted_confidence))
    
    return adjusted_confidence, reasons

    
def check_short_requirements(row, regime, structural_score, flow_score):
    """Check SHORT-specific requirements with nuanced flow confirmation"""
    reasons = []
    
    # Flow confirmation with nuance
    if flow_score == 0:
        # Allow only if structural weakness + bearish BTC context
        if not (
            structural_score > 0 and
            row.get('btc_ret_lag1', 0) < 0
        ):
            reasons.append("no_flow_confirmation")
    
    # R5 stronger flow requirement (only if we have flow signals)
    if regime == "R5" and flow_score > 0 and flow_score < 2:
        reasons.append("weak_distribution_flow")
    
    # Structural check - no structural weakness = no short
    if structural_score == 0:
        reasons.append("no_structural_break")
    
    # R3 short check (using new philosophy)
    if regime == "R3" and not check_r3_short_allowed(row):
        reasons.append("r3_no_early_weakness")
    
    return reasons

# ========== LONG-SPECIFIC LOGIC ==========
def long_veto(row):
    """LONG veto - minimal and asymmetric"""
    veto = []
    
    if row.get('btc_ret_lag1', 0) < -0.02:
        veto.append("btc_drawdown")
    
    if row.get('whale_exchange_flow_ratio', 0) > 0.6:
        veto.append("distribution")
    
    if row.get('eth_vol7', 0) > row.get('eth_vol30', 0) * 1.5:
        veto.append("vol_spike")
    
    return veto

# ========== SHADOW TRADING SYSTEM ==========
try:
    from shadow_trading import ShadowTrader
    print("✅ Modular shadow trading system imported successfully")
except ImportError as e:
    print(f"❌ Could not import from shadow_trading package: {e}")
    ShadowTrader = None

def long_confirmation(row):
    """
    LONG confirmation logic
    Stage A: ML finds accumulation
    Stage B: Confirm price is responding
    """
    # Updated to use vol_ratio
    vol_ratio = row.get('vol_ratio', 1)
    
    return (
        row.get('eth_ret_lag1', 0) > 0 and
        row.get('eth_ret_lag2', 0) > 0 and
        vol_ratio < 1.0  # Volatility compression (accumulation)
    )
# ========== MODEL MANAGEMENT ==========

def rebuild_models_if_needed(df_pipeline):
    """
    Rebuild models if they don't exist or feature mismatch
    Returns: (short_model, long_model)
    """
    short_model = None
    long_model = None
    
    # Check SHORT model
    short_model_path = 'models/r5_short_final.pkl'
    if os.path.exists(short_model_path):
        try:
            short_model = joblib.load(short_model_path)
            print(f"✅ SHORT model loaded from {short_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(short_model, 'feature_names_'):
                print(f"   Model expects {len(short_model.feature_names_)} features")
                missing = [f for f in short_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  SHORT model missing features: {missing[:5]}")
                    print("   Rebuilding SHORT model...")
                    short_model = None
            else:
                print("⚠️  SHORT model missing feature_names_, rebuilding...")
                short_model = None
        except Exception as e:
            print(f"⚠️  Error loading SHORT model: {e}")
            short_model = None
    
    # Check LONG model
    long_model_path = 'models/R1_R2_LONG.pkl'
    if os.path.exists(long_model_path):
        try:
            long_model = joblib.load(long_model_path)
            print(f"✅ LONG model loaded from {long_model_path}")
            
            # Check if model has feature_names_ attribute
            if hasattr(long_model, 'feature_names_'):
                print(f"   Model expects {len(long_model.feature_names_)} features")
                missing = [f for f in long_model.feature_names_ if f not in df_pipeline.columns]
                if missing:
                    print(f"⚠️  LONG model missing features: {missing[:5]}")
                    print("   Rebuilding LONG model...")
                    long_model = None
            else:
                print("⚠️  LONG model missing feature_names_, rebuilding...")
                long_model = None
        except Exception as e:
            print(f"⚠️  Error loading LONG model: {e}")
            long_model = None
    
    # Rebuild SHORT model if needed
    if short_model is None:
        print("\n" + "="*70)
        print("REBUILDING SHORT MODEL (R5)")
        print("="*70)
        
        df_r5 = df_pipeline[df_pipeline['regime_code'] == 'R5'].copy()
        
        # ✅ CRITICAL FIX 1: Use only features that exist in the data
        short_features = [f for f in SHORT_FEATURES if f in df_r5.columns]
        print(f"   Using {len(short_features)} SHORT features: {short_features}")
        
        if len(df_r5) >= 50:
            split_idx = int(len(df_r5) * 0.8)
            X_train_short = df_r5[short_features].iloc[:split_idx].fillna(0)
            y_train_short = df_r5['y_short_t2'].iloc[:split_idx]
            
            print(f"   Training on {len(X_train_short)} R5 samples")
            
            short_model = GradientBoostingClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42
            )
            short_model.fit(X_train_short, y_train_short)
            
            # ✅ CRITICAL FIX 2: Store feature names in the model
            short_model.feature_names_ = short_features
            joblib.dump(short_model, short_model_path)
            
            # Verify
            print(f"✅ SHORT model rebuilt and saved")
            print(f"   Model now expects {len(short_model.feature_names_)} features")
            print(f"   Features: {short_model.feature_names_}")
        else:
            print("⚠️  Insufficient R5 data for SHORT model")
    
    # Rebuild LONG model if needed
    if long_model is None:
        print("\n" + "="*70)
        print("REBUILDING LONG MODEL (R1+R2)")
        print("="*70)
        
        df_long = df_pipeline[df_pipeline['regime_code'].isin(['R1', 'R2'])].copy()
        
        # Remove obvious traps
        df_long = df_long[
            (df_long['btc_ret_lag1'] > -0.02) &
            (df_long['whale_exchange_flow_ratio'] < 0.6)
        ]
        
        # ✅ CRITICAL FIX 3: Use only features that exist in the data
        long_features = [f for f in LONG_FEATURES if f in df_long.columns]
        print(f"   Using {len(long_features)} LONG features: {long_features}")
        
        X_long = df_long[long_features].fillna(0)
        y_long = df_long['y_long_t2']
        
        if len(X_long) >= 50:
            long_model = GradientBoostingClassifier(
                n_estimators=120,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                random_state=42
            )
            
            long_model.fit(X_long, y_long)
            
            # ✅ CRITICAL FIX 4: Store feature names in the model
            long_model.feature_names_ = long_features
            joblib.dump(long_model, long_model_path)
            
            print("✅ LONG model rebuilt and saved")
            print(f"   Model now expects {len(long_model.feature_names_)} features")
            print(f"   Features: {long_model.feature_names_}")
            
            # Basic validation
            probs = long_model.predict_proba(X_long)[:, 1]
            preds = (probs >= 0.60).astype(int)
            prec = precision_score(y_long, preds, zero_division=0)
            rec = recall_score(y_long, preds, zero_division=0)
            
            print(f"   Training precision: {prec:.3f}")
            print(f"   Training recall: {rec:.3f}")
        else:
            print("⚠️  Insufficient LONG data")
    
    return short_model, long_model

def run_90_day_shadow_trading():
    """
    Run 90-day shadow trading with MAE/MFE logging
    Now using modular ShadowTrader from shadow_trading module
    """
    print("\n" + "="*70)
    print("90-DAY SHADOW TRADING INITIATED")
    print("="*70)
    print("Rules:")
    print("1. Execute NOTHING with real capital")
    print("2. Record every ENTER signal")
    print("3. Track MAE and MFE over T+48h")
    print("4. Answer the performance questions")
    print("="*70)
    
    # Check if shadow trading module is available
    if ShadowTrader is None:
        print("❌ Shadow trading module not available")
        print("   Please ensure shadow_trading/ directory exists with shadow_trader.py")
        return []
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return []
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return []
    
    # Initialize shadow trader from modular system
    trader = ShadowTrader()
    
    # SIMPLIFIED: Just use the last 90 days of available data
    # We'll use a smaller forward window if data is limited
    available_days = len(df)
    
    if available_days < 90:
        print(f"⚠️  Limited data: Only {available_days} days available")
        print(f"   Using all available data with reduced forward window")
        test_days = available_days
        forward_window = min(30, available_days // 3)  # Use 30 days or 1/3 of data
    else:
        test_days = 90
        forward_window = 48
    
    # Get test period (exclude forward window)
    start_idx = max(0, len(df) - test_days - forward_window)
    end_idx = len(df) - forward_window
    
    if start_idx >= end_idx:
        print(f"❌ Insufficient data for shadow trading")
        print(f"   Need at least {forward_window + 1} days")
        return []
    
    test_period = df.iloc[start_idx:end_idx].copy()
    
    print(f"\n📅 Running shadow trading on {len(test_period)} days:")
    print(f"   Date range: {test_period['block_date'].min().date()} to {test_period['block_date'].max().date()}")
    print(f"   Forward window: {forward_window} days")
    print(f"   Total data points: {len(df)}")
    
    # Generate and log signals
    signals_logged = 0
    for i, row in test_period.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        
        # Log ENTER signals only
        if signal['action'] == 'ENTER':
            trade = trader.log_trade(signal, df, trade_days=forward_window)
            if trade:
                signals_logged += 1
                direction_icon = "🟢" if signal['direction'] == 'LONG' else "🔴"
                print(f"{direction_icon} Logged {signal['date']}: {signal['direction']} @ ${row['eth_price']:.0f} "
                      f"(conf: {signal['adjusted_confidence']:.2f}, size: {signal['position_size']:.2f})")
    
    # Save and analyze
    if signals_logged > 0:
        trader.save_trades()
        
        # Get performance report from modular system
        try:
            report = trader.get_performance_report()
            if report and "error" not in report:
                print("\n📊 MODULAR PERFORMANCE REPORT:")
                print("-" * 70)
                print(f"Total trades: {report.get('total_trades', 0)}")
                print(f"Long trades: {report.get('long_trades', 0)}")
                print(f"Short trades: {report.get('short_trades', 0)}")
                print(f"Average MAE: {report.get('avg_mae', 0):.2f}%")
                print(f"Average MFE: {report.get('avg_mfe', 0):.2f}%")
                print(f"Average return: {report.get('avg_final_return', 0):.2f}%")
                print(f"Win rate: {report.get('win_rate', 0)*100:.1f}%")
                print(f"Liquidation risk (3x): {report.get('liquidation_risk_3x', 0)} trades")
                
                # Show regime distribution
                regime_dist = report.get('regime_distribution', {})
                if regime_dist:
                    print(f"\nRegime Distribution:")
                    for regime, count in regime_dist.items():
                        print(f"  {regime}: {count} trades")
                
                # Show best/worst trades if available
                if 'best_trade' in report:
                    best = report['best_trade']
                    print(f"\n🏆 Best trade: {best['date']} {best['direction']}: {best['return']:.2f}%")
                
                if 'worst_trade' in report:
                    worst = report['worst_trade']
                    print(f"💥 Worst trade: {worst['date']} {worst['direction']}: {worst['return']:.2f}%")
                
                # Sharpe ratio if available
                if 'sharpe_ratio' in report:
                    print(f"📈 Sharpe ratio: {report['sharpe_ratio']:.2f}")
            else:
                print("⚠️  Could not generate performance report")
        except Exception as e:
            print(f"⚠️  Error generating performance report: {e}")
        
        # MAE sanity check question (STEP 3)
        print("\n" + "="*70)
        print("MAE SANITY CHECK (STEP 3)")
        print("="*70)
        print("Question: Did any winning signals experience MAE large enough")
        print("to liquidate a reasonable position?")
        print("-" * 70)
        
        df_trades = pd.DataFrame(trader.trades)
        liquidation_threshold = -33.33  # 3x leverage liquidation
        
        risky_trades = df_trades[df_trades['mae_pct'] < liquidation_threshold]
        
        if len(risky_trades) > 0:
            print(f"⚠️  YES: {len(risky_trades)} trades would liquidate at 3x leverage")
            print("\nRisky trades:")
            for _, trade in risky_trades.iterrows():
                print(f"  {trade['entry_date']} {trade['direction']}: MAE = {trade['mae_pct']:.2f}%")
            print("\n💡 Consider: MAE-aware sizing cap or leverage constraint")
        else:
            print("✅ NO: All trades have MAE > -33.33%")
            print("   No immediate need for MAE-aware sizing")
        
        # Performance questions (STEP 5)
        print("\n" + "="*70)
        print("PERFORMANCE QUESTIONS (STEP 5)")
        print("="*70)
        
        # Question 1: Are SHORT signals rare but violent?
        shorts = df_trades[df_trades['direction'] == 'SHORT']
        if len(shorts) > 0:
            avg_short_mfe = shorts['mfe_pct'].mean()
            is_violent = avg_short_mfe > 5.0  # >5% average MFE
            q1 = "YES" if is_violent else "Needs review"
            print(f"1. Are SHORT signals rare but violent?")
            print(f"   - SHORT count: {len(shorts)}/{len(df_trades)} ({len(shorts)/len(df_trades)*100:.1f}%)")
            print(f"   - Average SHORT MFE: {avg_short_mfe:.2f}%")
            print(f"   - Answer: {q1}\n")
        else:
            q1 = "NO SHORTS"
            print(f"1. Are SHORT signals rare but violent?")
            print(f"   - No SHORT signals recorded\n")
        
        # Question 2: Do SHORTs cluster near distribution/breakdowns?
        if len(shorts) > 0:
            r5_shorts = shorts[shorts['regime'] == 'R5']
            q2_ratio = len(r5_shorts) / len(shorts) if len(shorts) > 0 else 0
            q2 = "YES" if q2_ratio > 0.5 else "Mixed"
            print(f"2. Do SHORTs cluster near distribution/breakdowns?")
            print(f"   - R5 SHORTs: {len(r5_shorts)}/{len(shorts)} ({q2_ratio*100:.1f}%)")
            print(f"   - Answer: {q2}\n")
        else:
            q2 = "NO DATA"
            print(f"2. Do SHORTs cluster near distribution/breakdowns?")
            print(f"   - No SHORT signals to analyze\n")
        
        # Question 3: Are rejected signals obviously bad in hindsight?
        print(f"3. Are rejected signals obviously bad in hindsight?")
        print(f"   - Manual review needed: Check NO_TRADE days vs price action\n")
        
        # Question 4: Does higher confidence → better MFE?
        if len(df_trades) >= 10:
            corr = df_trades['confidence'].corr(df_trades['mfe_pct'])
            q4 = "YES" if corr > 0.1 else "Weak relationship"
            print(f"4. Does higher confidence → better MFE?")
            print(f"   - Correlation: {corr:.3f}")
            print(f"   - Answer: {q4}\n")
        else:
            q4 = "INSUFFICIENT DATA"
            print(f"4. Does higher confidence → better MFE?")
            print(f"   - Need at least 10 trades, have {len(df_trades)}\n")
        
        # Count answers
        answers = [q1, q2, q4] if len(shorts) > 0 and len(df_trades) >= 10 else []
        yes_count = sum(1 for a in answers if 'YES' in str(a))
        
        print(f"📈 SCORE: {yes_count}/{len(answers)} performance questions positive")
        if yes_count >= 3:
            print("✅ CONCLUSION: Core system is production-grade")
        elif yes_count >= 2:
            print("⚠️  CONCLUSION: System promising but needs minor adjustments")
        elif yes_count >= 1:
            print("⚠️  CONCLUSION: System needs significant review")
        else:
            print("❌ CONCLUSION: System not ready for production")
            
    else:
        print("⚠️  No ENTER signals logged in shadow trading period")
        print("   This may be correct behavior (selective system)")
        print("   Check if this matches expectations")
        
        # Still save empty trades file for tracking
        try:
            trader.save_trades()
        except:
            print("   Could not save empty trades file")
    
    # Return trades for further analysis
    return trader.trades if hasattr(trader, 'trades') else []

def check_shadow_trading_files():
    """Check if shadow trading files were created"""
    print("\n" + "="*70)
    print("SHADOW TRADING FILES CHECK")
    print("="*70)
    
    files_to_check = [
        'shadow_trading/shadow_trades_90d.csv',
        'shadow_trading/shadow_analysis.md'
    ]
    
    files_found = 0
    for filepath in files_to_check:
        if os.path.exists(filepath):
            size = os.path.getsize(filepath)
            print(f"✅ {filepath}: {size} bytes")
            files_found += 1
            
            # Show first few lines for CSV
            if filepath.endswith('.csv'):
                try:
                    df = pd.read_csv(filepath)
                    print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")
                    if len(df) > 0:
                        print(f"   First trade: {df.iloc[0]['entry_date']}")
                        print(f"   Last trade: {df.iloc[-1]['entry_date']}")
                except Exception as e:
                    print(f"   Could not read CSV: {e}")
        else:
            print(f"❌ {filepath}: Not found")
    
    # Check modular system files
    print("\n📁 Modular System Check:")
    module_files = [
        'shadow_trading/shadow_analysis.py',
        'shadow_trading/shadow_trader.py',
        'shadow_trading/__init__.py'
    ]
    
    for filepath in module_files:
        if os.path.exists(filepath):
            print(f"✅ {filepath}: Found")
        else:
            print(f"❌ {filepath}: Missing")
    
    if files_found == len(files_to_check):
        print(f"\n✅ All shadow trading files present and accounted for!")
    else:
        print(f"\n⚠️  {files_found}/{len(files_to_check)} output files found")

# ========== UNIFIED SIGNAL GENERATION ==========

def generate_unified_signal(row, df, short_model, long_model):
    """
    UNIFIED SIGNAL GENERATION with optional funding modifier
    """
    regime = row.get('regime_code', 'R0')
    date_str = str(row['block_date'].date()) if 'block_date' in row else str(row.name)
    
    # Base signal object
    signal = {
        "date": date_str,
        "regime": regime,
        "direction": None,
        "model_probability": 0.0,
        "adjusted_confidence": 0.0,
        "position_size": 0.0,
        "reasons": [],
        "funding_available": False,
        "funding_rate": None,
        "action": "NO_TRADE"
    }
    
    # Record if funding data exists for this row
    if 'eth_funding_rate_8h' in row.index and not pd.isna(row.get('eth_funding_rate_8h')):
        signal["funding_available"] = True
        signal["funding_rate"] = float(row['eth_funding_rate_8h'])
    
    # ===== LONG LOGIC (R1/R2/R3) =====
    if regime in ['R1', 'R2', 'R3'] and long_model:
        signal["direction"] = "LONG"
        
        # Use model's stored feature names for inference
        if not hasattr(long_model, 'feature_names_'):
            signal["reasons"] = ["model_error: long_model missing feature_names_"]
            return signal
        
        features = long_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = long_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold
        if prob < LONG_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # Step 2: Calculate confirmation score
        confirm_score = 0

        if row.get('eth_ret_lag1', 0) > 0:
            confirm_score += 1
        if row.get('eth_ret_lag2', 0) > 0:
            confirm_score += 1
        if row.get('vol_ratio', 1) < 1.0:
            confirm_score += 1
        
        # ✅ FIX 1: Define required_score based on regime
        if regime == "R1":
            required_score = 1  # Early bull: allow early signs
        elif regime == "R2":
            required_score = 2  # Late bull: require agreement
        elif regime == "R3":
            required_score = 3  # Early bear: extremely strict
        else:
            required_score = 2  # Safe default
        
        if confirm_score < required_score:
            signal["reasons"] = [f"weak_price_confirmation ({confirm_score}/{required_score})"]
            return signal
        
        # Step 3: Apply LONG vetoes
        veto_reasons = long_veto(row)
        if veto_reasons:
            signal["reasons"] = veto_reasons
            return signal
        
        # Step 4: Calculate confidence WITH OPTIONAL FUNDING MODIFIER
        veto_score = len(veto_reasons)
        adj_conf, funding_reasons = adjust_confidence_unified(
            prob, regime, direction="LONG", veto_score=veto_score, row=row
        )
        signal["adjusted_confidence"] = adj_conf
        
        # Add funding reasons to signal if any
        if funding_reasons:
            signal["reasons"].extend(funding_reasons)
        
        # Step 5: Regime-aware confidence floor
        if regime == "R1":
            confidence_floor = 0.50  # Lower for early bull
        elif regime == "R2":
            confidence_floor = 0.55  # Higher for late bull
        elif regime == "R3":
            confidence_floor = 0.60  # Highest for early bear (should be rare)
        else:
            confidence_floor = 0.55  # Default
        
        if signal["adjusted_confidence"] < confidence_floor:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        # ✅ FIXED: Use the updated position sizing function
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="LONG"
        )
        signal["reasons"] = ["ml_accumulation", "price_confirmation"]
        signal["action"] = "ENTER"
        return signal
        
    # ===== SHORT LOGIC (R3/R5) =====
          
    elif regime in ['R3', 'R5'] and short_model:
        signal["direction"] = "SHORT"
        
        # Use model's stored feature names for inference
        if not hasattr(short_model, 'feature_names_'):
            signal["reasons"] = ["model_error: short_model missing feature_names_"]
            return signal
        
        features = short_model.feature_names_
        try:
            # Ensure features exist in row and fill missing with 0
            X = row.reindex(features, fill_value=0).values.reshape(1, -1)
            prob = short_model.predict_proba(X)[0, 1]
            signal["model_probability"] = float(prob)
        except Exception as e:
            signal["reasons"] = [f"model_error: {str(e)[:100]}"]
            return signal
        
        # Step 1: Check probability threshold (HIGHER for SHORT)
        if prob < SHORT_ENTRY_THRESHOLD:
            signal["reasons"] = ["low_model_probability"]
            return signal
        
        # Step 2: Calculate veto scores
        veto, structural_score, flow_score, context_score, veto_reasons = calculate_short_veto_score(row)
        
        # Step 3: Check SHORT-specific requirements
        requirement_failures = check_short_requirements(row, regime, structural_score, flow_score)
        if requirement_failures:
            signal["reasons"] = requirement_failures
            return signal
        
        # Step 4: Calculate confidence WITH OPTIONAL FUNDING MODIFIER
        adj_conf, funding_reasons = adjust_confidence_unified(
            prob, regime, direction="SHORT", veto_score=veto, row=row
        )
        signal["adjusted_confidence"] = adj_conf
        
        # Add funding reasons if any
        if funding_reasons:
            signal["reasons"].extend(funding_reasons)
        
        # Final confidence check
        if signal["adjusted_confidence"] < 0.55:
            signal["reasons"] = ["low_final_confidence"]
            signal["direction"] = None
            return signal
        
        signal["position_size"] = map_confidence_to_size_unified(
            signal["adjusted_confidence"], regime, direction="SHORT"
        )
        signal["reasons"] = veto_reasons  # Use veto reasons as trade reasons
        signal["action"] = "ENTER"
        return signal
    
    # ===== NEUTRAL REGIME =====
    else:
        signal["reasons"] = ["neutral_regime"]
        return signal
            
def generate_daily_signal_unified(df, short_model, long_model):
    """
    Generate unified daily signal (uses latest row)
    """
    latest_row = df.iloc[-1].copy()
    return generate_unified_signal(latest_row, df, short_model, long_model)

# ========== SIGNAL INSPECTION ==========
def inspect_signals_unified(df, short_model, long_model, num_signals=60):
    """
    Inspect signals with optional funding modifier
    """
    print("\n" + "="*70)
    print(f"UNIFIED SIGNAL INSPECTION (Last {num_signals} days)")
    print("="*70)
    
    # Count funding availability
    funding_available = 0
    if 'eth_funding_rate_8h' in df.columns:
        recent = df.iloc[-num_signals:]
        funding_available = recent['eth_funding_rate_8h'].notna().sum()
        print(f"Funding data available: {funding_available}/{num_signals} days ({funding_available/num_signals*100:.1f}%)")
    
    recent_data = df.iloc[-num_signals:].copy()
    print(f"Date range: {recent_data['block_date'].min().date()} to {recent_data['block_date'].max().date()}")
    
    signals = []
    long_count = 0
    short_count = 0
    no_trade_count = 0
    
    for idx, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print only trade signals
        if signal['action'] == 'ENTER':
            if signal['direction'] == 'LONG':
                long_count += 1
                funding_status = f" | Funding: {signal.get('funding_rate', 0)*100:.3f}%" if signal.get('funding_rate') is not None else ""
                print(f"\n🟢 LONG:  {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}{funding_status}")
            elif signal['direction'] == 'SHORT':
                short_count += 1
                funding_status = f" | Funding: {signal.get('funding_rate', 0)*100:.3f}%" if signal.get('funding_rate') is not None else ""
                print(f"\n🔴 SHORT: {signal['date']} | Regime: {signal['regime']} | "
                      f"Conf: {signal['adjusted_confidence']:.2f} | Size: {signal['position_size']:.2f}{funding_status}")
        else:
            no_trade_count += 1
    
    print(f"\n📊 Signal Summary:")
    print(f"   Total days: {len(signals)}")
    print(f"   LONG signals: {long_count} ({long_count/len(signals)*100:.1f}%)")
    print(f"   SHORT signals: {short_count} ({short_count/len(signals)*100:.1f}%)")
    print(f"   NO_TRADE: {no_trade_count} ({no_trade_count/len(signals)*100:.1f}%)")
    
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = {}
    for signal in signals:
        regime = signal.get('regime', 'UNKNOWN')
        regime_dist[regime] = regime_dist.get(regime, 0) + 1
    
    for regime in sorted(regime_dist.keys()):
        count = regime_dist[regime]
        print(f"   {regime}: {count} days ({count/len(signals)*100:.1f}%)")
    
    # Rejection reasons analysis
    print(f"\n🔍 Rejection Reasons:")
    rejection_reasons = {}
    for signal in signals:
        if signal['action'] == 'NO_TRADE' and signal.get('reasons'):
            for reason in signal['reasons']:
                rejection_reasons[reason] = rejection_reasons.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"   {reason}: {count} times")
    
    return signals

def inspect_long_signals_bull_cycle(df, short_model, long_model, start_date='2020-01-01', end_date='2022-01-01'):
    """
    Inspect LONG signals during the 2020-2021 bull cycle
    """
    print("\n" + "="*70)
    print(f"LONG SIGNAL INSPECTION: {start_date} to {end_date}")
    print("="*70)
    
    # Filter to bull cycle period
    mask = (df['block_date'] >= start_date) & (df['block_date'] <= end_date)
    bull_data = df[mask].copy()
    
    print(f"Period: {bull_data['block_date'].min().date()} to {bull_data['block_date'].max().date()}")
    print(f"Total days: {len(bull_data)}")
    
    # Get R1/R2 days
    bull_regimes = bull_data[bull_data['regime_code'].isin(['R1', 'R2'])].copy()
    print(f"R1/R2 days: {len(bull_regimes)}")
    print(f"  R1: {(bull_regimes['regime_code'] == 'R1').sum()} days")
    print(f"  R2: {(bull_regimes['regime_code'] == 'R2').sum()} days")
    
    # Generate signals for R1/R2 days only
    long_signals = []
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['direction'] == 'LONG':
            long_signals.append(signal)
    
    # Analyze LONG signals
    print(f"\n📊 LONG Signal Analysis:")
    print(f"  Total LONG signals: {len(long_signals)}")
    
    if len(long_signals) > 0:
        df_long_signals = pd.DataFrame(long_signals)
        
        # Group by regime
        regime_counts = df_long_signals['regime'].value_counts()
        for regime, count in regime_counts.items():
            print(f"  {regime}: {count} signals")
        
        # Analyze timing (early vs late bull)
        df_long_signals['date_dt'] = pd.to_datetime(df_long_signals['date'])
        df_long_signals['month'] = df_long_signals['date_dt'].dt.to_period('M')
        monthly_counts = df_long_signals['month'].value_counts().sort_index()
        
        print(f"\n📅 Monthly distribution:")
        for month, count in monthly_counts.head(12).items():  # Show first 12 months
            print(f"  {month}: {count} signals")
        
        # Check if signals avoid tops
        print(f"\n🔍 Top avoidance check:")
        for signal in df_long_signals.head(5).to_dict('records'):  # Show first 5 signals
            date_str = signal['date']
            row = bull_data[bull_data['block_date'] == pd.Timestamp(date_str)]
            if not row.empty:
                row = row.iloc[0]
                # Check if price near highs (should be False for good LONGs)
                near_highs = not price_not_near_highs(row, df, lookback=90, max_pct=0.75)
                status = "⚠️ NEAR HIGHS" if near_highs else "✅ NOT NEAR HIGHS"
                print(f"  {date_str}: {status} | Price: ${row['eth_price']:.0f} | Conf: {signal['adjusted_confidence']:.2f}")
    
    # Also check how many R1/R2 days were rejected and why
    print(f"\n🔍 LONG Rejection Analysis (R1/R2 days):")
    rejection_counts = {}
    for idx, row in bull_regimes.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        if signal['action'] != 'ENTER' and signal['direction'] == 'LONG':
            for reason in signal['reasons']:
                rejection_counts[reason] = rejection_counts.get(reason, 0) + 1
    
    for reason, count in sorted(rejection_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {reason}: {count} times")
    
    return long_signals

def daily_execution_workflow():
    """
    Formal daily execution workflow (Step 1)
    """
    print("\n" + "="*70)
    print("DAILY EXECUTION WORKFLOW")
    print("="*70)
    print("🕘 Execution Order:")
    print("   1. Pull data (T-1 close)")
    print("   2. On-chain features")
    print("   3. Price / volatility")
    print("   4. Funding rate (veto modifier only)")
    print("   5. Assign regime")
    print("   6. Generate unified signal")
    print("   7. Apply confidence → size mapping")
    print("   8. Decide action (ENTER/NO_TRADE)")
    print("   9. Log everything")
    print("="*70)
    
    # Run the pipeline
    df_pipeline, short_model, long_model = run_unified_pipeline()
    
    if df_pipeline is not None:
        print("\n✅ Daily execution complete")
        print("\n📋 Output contract:")
        with open('data/latest_signal_unified.json', 'r') as f:
            signal = json.load(f)
            print(json.dumps(signal, indent=2))
        
        # Log to daily log file
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "signal": signal,
            "execution_status": "completed"
        }
        
        os.makedirs('logs/daily', exist_ok=True)
        log_file = f"logs/daily/{datetime.now().strftime('%Y-%m-%d')}.json"
        with open(log_file, 'w') as f:
            json.dump(log_entry, f, indent=2)
        
        print(f"\n📝 Log saved: {log_file}")

# ========== MAIN PIPELINE ==========
def run_unified_pipeline():
    """
    Execute complete pipeline with funding data from loader
    """
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("✅ Features FROZEN")
    print("✅ Regimes FROZEN")
    print("✅ Funding: Veto-only from data loader")
    print("✅ Volatility: vol_ratio feature")
    print("="*70)
    
    # Step 1: Load all data including funding
    df_whales, df_market, df_btc, df_eth, df_funding = load_data_from_files()
    
    # Check essential data
    if any(d.empty for d in [df_whales, df_market, df_btc, df_eth]):
        print("\n❌ Missing essential data. Run data_loader.py first.")
        return None, None, None
    
    # Step 2: Engineer features with funding data
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth, df_funding)
    
    # Step 3: Build pipeline
    df_pipeline = build_pipeline_complete(df_features)    
    # Step 4: Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df_pipeline)
    
    # Step 5: Generate live signal
    if short_model and long_model:
        print("\n" + "="*70)
        print("GENERATING UNIFIED LIVE SIGNAL")
        print("="*70)
        
        signal = generate_daily_signal_unified(df_pipeline, short_model, long_model)
        print(json.dumps(signal, indent=2))
        
        with open('data/latest_signal_unified.json', 'w') as f:
            json.dump(signal, f, indent=2)
    
    # Step 6: Inspect 60-day history
    if short_model and long_model:
        print("\n" + "="*70)
        print("60-DAY UNIFIED SIGNAL INSPECTION")
        print("="*70)
        
        signals = inspect_signals_unified(df_pipeline, short_model, long_model, 60)
        
        df_signals = pd.DataFrame(signals)
        df_signals.to_csv('validation/signal_inspection_unified.csv', index=False)
        print(f"\n✅ Saved: validation/signal_inspection_unified.csv")
    
    return df_pipeline, short_model, long_model

# ========== PAPER TRADE TEST ==========
def run_unified_paper_test():
    """
    Run 60-day paper trade test with UNIFIED logic
    """
    print("\n" + "="*70)
    print("60-DAY PAPER TRADE TEST (UNIFIED LOGIC)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Take last 60 days
    test_period = df.iloc[-60:].copy()
    
    # Generate signals
    signals = []
    for i, row in test_period.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
        
        # Print trade signals
        if signal['action'] == 'ENTER':
            direction_icon = "🟢" if signal['direction'] == 'LONG' else "🔴"
            print(f"{direction_icon} {signal['date']}: {signal['action']} {signal['direction']} "
                  f"@ ${row['eth_price']:.0f} (conf: {signal['adjusted_confidence']:.2f}, "
                  f"size: {signal['position_size']:.2f})")
    
    # Analyze results
    df_signals = pd.DataFrame(signals)
    
    print(f"\n📊 Test Results (60 days):")
    print(f"   Total days: {len(df_signals)}")
    print(f"   ENTER signals: {(df_signals['action'] == 'ENTER').sum()}")
    print(f"   LONG signals: {(df_signals['direction'] == 'LONG').sum()}")
    print(f"   SHORT signals: {(df_signals['direction'] == 'SHORT').sum()}")
    
    # Manual review questions
    if (df_signals['action'] == 'ENTER').sum() > 0:
        print(f"\n🔍 Manual Review Questions:")
        print(f"   1. Do LONGs occur only in R1/R2?")
        print(f"   2. Do SHORTs occur only in R3/R5?")
        print(f"   3. Are LONGs buying strength, not tops?")
        print(f"   4. Are SHORTs selling weakness/distribution?")
        print(f"   5. Are position sizes reasonable for regime?")
    
    # Regime distribution
    print(f"\n📈 Regime Distribution:")
    regime_dist = test_period['regime_code'].value_counts()
    for regime, count in regime_dist.items():
        print(f"   {regime}: {count} days ({count/len(test_period)*100:.1f}%)")
    
    return df_signals

# ========== MANUAL SIGNAL REVIEW ==========
def manual_signal_review(num_days=30):
    """
    Manual review of signals with guided questions
    """
    print("\n" + "="*70)
    print(f"MANUAL SIGNAL REVIEW ({num_days} days)")
    print("="*70)
    
    # Load pipeline data
    if not os.path.exists('data/pipeline_complete.csv'):
        print("❌ Pipeline data not found. Run unified pipeline first.")
        return
    
    df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
    df = df.sort_values('block_date')
    
    # Rebuild models if needed
    short_model, long_model = rebuild_models_if_needed(df)
    
    if not short_model or not long_model:
        print("❌ Could not load or rebuild models")
        return
    
    # Generate signals
    recent_data = df.iloc[-num_days:].copy()
    signals = []
    
    for i, row in recent_data.iterrows():
        signal = generate_unified_signal(row, df, short_model, long_model)
        signals.append(signal)
    
    # Review template
    print("\n📝 Review Template (for each ENTER signal):")
    print("-" * 40)
    
    for signal in signals:
        if signal['action'] == 'ENTER':
            print(f"\n📅 {signal['date']} - {signal['direction']} in {signal['regime']}")
            print(f"   Confidence: {signal['adjusted_confidence']:.2f}")
            print(f"   Position Size: {signal['position_size']:.2f}")
            print(f"   Reasons: {', '.join(signal['reasons'])}")
            
            # Find the row data
            row = df[df['block_date'] == pd.Timestamp(signal['date'])]
            if not row.empty:
                row = row.iloc[0]
                print(f"   ETH Price: ${row['eth_price']:.0f}")
                print(f"   BTC Ret Lag1: {row.get('btc_ret_lag1', 0):.3f}")
                print(f"   ETH Ret Lag1: {row.get('eth_ret_lag1', 0):.3f}")
            
            # Review questions
            if signal['direction'] == 'LONG':
                print("   Questions:")
                print("   1. Is price breaking structure upward?")
                print("   2. Is BTC aligned or neutral?")
                print("   3. Are we buying strength, not tops?")
            else:
                print("   Questions:")
                print("   1. Is this distribution or panic?")
                print("   2. Is liquidity present?")
                print("   3. Is this early weakness (R3) or real distribution (R5)?")
            
            print("-" * 40)
    
    return signals
def verify_vol_ratio_integration():
    """Verify vol_ratio is properly integrated"""
    print("\n" + "="*70)
    print("VOL_RATIO INTEGRATION VERIFICATION")
    print("="*70)
    
    # Load data
    df_whales, df_market, df_btc, df_eth = load_data_from_files()
    df_features = engineer_features(df_whales, df_market, df_btc, df_eth)
    
    print("\n📊 Vol Ratio Statistics:")
    if 'vol_ratio' in df_features.columns:
        print(f"✅ vol_ratio column exists")
        print(f"   Mean: {df_features['vol_ratio'].mean():.3f}")
        print(f"   Min: {df_features['vol_ratio'].min():.3f}")
        print(f"   Max: {df_features['vol_ratio'].max():.3f}")
        print(f"   < 1.0 (compression): {(df_features['vol_ratio'] < 1.0).sum()} days")
        print(f"   > 1.0 (expansion): {(df_features['vol_ratio'] > 1.0).sum()} days")
        
        # Check recent values
        recent = df_features.tail(10)
        print(f"\n📅 Recent vol_ratio values:")
        for i, row in recent.iterrows():
            date = row['block_date'].date()
            vol_ratio = row['vol_ratio']
            eth_vol7 = row.get('eth_vol7', 0)
            eth_vol30 = row.get('eth_vol30', 1)
            status = "📉 COMPRESSION" if vol_ratio < 1.0 else "📈 EXPANSION"
            print(f"   {date}: {vol_ratio:.3f} ({status}) | 7d:{eth_vol7:.4f} / 30d:{eth_vol30:.4f}")
    else:
        print("❌ vol_ratio column missing!")
    
    # Check if models will use vol_ratio
    print(f"\n🔍 Feature List Check:")
    print(f"   vol_ratio in LONG_FEATURES: {'vol_ratio' in LONG_FEATURES}")
    print(f"   vol_ratio in SHORT_FEATURES: {'vol_ratio' in SHORT_FEATURES}")
    
    # Build pipeline to check model rebuilding
    df_pipeline = build_pipeline_complete(df_features)
    short_model, long_model = rebuild_models_if_needed(df_pipeline)
    
    if short_model and hasattr(short_model, 'feature_names_'):
        print(f"\n✅ SHORT model features:")
        print(f"   {short_model.feature_names_}")
        print(f"   Uses vol_ratio: {'vol_ratio' in short_model.feature_names_}")
    
    if long_model and hasattr(long_model, 'feature_names_'):
        print(f"\n✅ LONG model features:")
        print(f"   {long_model.feature_names_}")
        print(f"   Uses vol_ratio: {'vol_ratio' in long_model.feature_names_}")

def check_file_system():
    """
    Check file system permissions and directories
    """
    print("\n" + "="*70)
    print("FILE SYSTEM CHECK")
    print("="*70)
    
    directories_to_check = [
        'shadow_trading',
        'validation',
        'backtest',
        'models',
        'data',
        'logs/daily'
    ]
    
    for directory in directories_to_check:
        try:
            os.makedirs(directory, exist_ok=True)
            # Check if writable
            test_file = os.path.join(directory, 'test_write.tmp')
            with open(test_file, 'w') as f:
                f.write('test')
            os.remove(test_file)
            print(f"✅ {directory}: Writable")
        except Exception as e:
            print(f"❌ {directory}: Not writable - {str(e)[:100]}")
    
    # Check current working directory
    print(f"\n📁 Current directory: {os.getcwd()}")
    print(f"📁 Directory contents: {len(os.listdir('.'))} items")
    
    # List shadow trading directory if it exists
    if os.path.exists('shadow_trading'):
        print(f"📁 shadow_trading contents: {os.listdir('shadow_trading')}")
        
def test_shadow_trading_basic():
    """
    Quick test of shadow trading system
    """
    print("\n🧪 Testing Shadow Trading System...")
    
    # Create a simple test dataframe
    dates = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
    test_df = pd.DataFrame({
        'block_date': dates,
        'eth_price': np.random.uniform(2000, 4000, len(dates))
    })
    
    # Create a test signal
    test_signal = {
        'date': '2025-06-15',
        'action': 'ENTER',
        'direction': 'SHORT',
        'regime': 'R5',
        'position_size': 1.0,
        'adjusted_confidence': 0.75,
        'model_probability': 0.80,
        'reasons': ['test_reason'],
        'funding_rate': 0.0001,
        'funding_available': True
    }
    
    # Test ShadowTrader
    trader = ShadowTrader()
    trade = trader.log_trade(test_signal, test_df, trade_days=10)
    
    if trade:
        print(f"✅ Basic test passed!")
        print(f"   Trade logged: {trade['entry_date']} {trade['direction']}")
        print(f"   MAE: {trade['mae_pct']:.2f}%")
        print(f"   MFE: {trade['mfe_pct']:.2f}%")
    else:
        print(f"❌ Basic test failed")
        print(f"   Check if dates match: {test_signal['date']} in {test_df['block_date'].min()} to {test_df['block_date'].max()}")
    
    # Test saving
    if trader.trades:
        trader.save_trades()
        print(f"✅ Save test passed!")
    else:
        print(f"❌ No trades to save")
    
    return trader.trades

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    print("\n" + "="*70)
    print("ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT")
    print("="*70)
    print("PHASE: 90-DAY SHADOW TRADING")
    print(f"{'='*70}")
    
    # First check file system
    check_file_system()
    
    required_files = [
        'data/whale_ml_ready.csv',
        'data/market_intent_ml_ready.csv', 
        'data/price_cache/btc.csv',
        'data/price_cache/eth.csv'
    ]
    
    missing_files = [f for f in required_files if not os.path.exists(f)]
    
    if missing_files:
        print(f"\n⚠️  Missing data files:")
        for f in missing_files:
            print(f"   - {f}")
        print(f"\nPlease run data_loader.py first to fetch data")
        print(f"Or place the required CSV files in the data directory")
        exit(1)
    
        # Ask user what to do
    print("\n📋 Available Options:")
    print("   1. Run unified pipeline (train models + generate signal)")
    print("   2. 60-day paper trade test (unified logic)")
    print("   3. Manual signal review (30 days)")
    print("   4. Inspect signals (60 days)")
    print("   5. Load and check data only")
    print("   6. Extended LONG inspection (2020-2021 bull cycle)")
    print("   7. [NEW] 90-day shadow trading with MAE/MFE")
    print("   8. [DEBUG] Test shadow trading system")  
        
    choice = input("\nSelect option (1-8): ").strip()
    
    if choice == '1':
        df_pipeline, short_model, long_model = run_unified_pipeline()
        
        if df_pipeline is not None:
            print("\n✅ Unified pipeline complete")
            print("\n📋 Next steps:")
            print("   1. Review validation/signal_inspection_unified.csv")
            print("   2. Run option 2 for paper trade test")
            print("   3. Run option 7 for 90-day shadow trading")
    
    elif choice == '2':
        signals = run_unified_paper_test()
        
        print("\n📋 Review questions answered:")
        print("   ✅ LONGs only in R1/R2?")
        print("   ✅ SHORTs only in R3/R5?")
        print("   ✅ Position sizing consistent?")
        print("   ✅ Confidence ranges reasonable?")
    
    elif choice == '3':
        num_days = input("How many days to review? (default: 30): ").strip()
        try:
            num_days = int(num_days) if num_days else 30
        except:
            num_days = 30
        
        signals = manual_signal_review(num_days)
    
    elif choice == '4':
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            short_model, long_model = rebuild_models_if_needed(df)
            
            if short_model and long_model:
                signals = inspect_signals_unified(df, short_model, long_model, 60)
    
    elif choice == '5':
        print("\n📂 Loading and checking data...")
        df_whales, df_market, df_btc, df_eth = load_data_from_files()
        
        print(f"\n✅ Data loaded successfully:")
        print(f"   Whale data: {len(df_whales)} rows")
        print(f"   Market data: {len(df_market)} rows")
        print(f"   BTC price: {len(df_btc)} rows")
        print(f"   ETH price: {len(df_eth)} rows")
    
    elif choice == '6':
        print("\n" + "="*70)
        print("EXTENDED LONG INSPECTION (2020-2021 BULL CYCLE)")
        print("="*70)
        
        # Load pipeline data
        if not os.path.exists('data/pipeline_complete.csv'):
            print("❌ Pipeline data not found. Run option 1 first.")
        else:
            df = pd.read_csv('data/pipeline_complete.csv', parse_dates=['block_date'])
            df = df.sort_values('block_date')
            
            # Load models
            short_model_path = 'models/r5_short_final.pkl'
            long_model_path = 'models/R1_R2_LONG.pkl'
            
            if os.path.exists(short_model_path) and os.path.exists(long_model_path):
                short_model = joblib.load(short_model_path)
                long_model = joblib.load(long_model_path)
                
                # Ask for date range
                print("\n📅 Select inspection period:")
                print("   1. 2020-2021 bull cycle (2020-01-01 to 2022-01-01)")
                print("   2. Full available history")
                print("   3. Custom date range")
                
                period_choice = input("Select (1-3): ").strip()
                
                if period_choice == '1':
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                elif period_choice == '2':
                    start_date = df['block_date'].min().strftime('%Y-%m-%d')
                    end_date = df['block_date'].max().strftime('%Y-%m-%d')
                elif period_choice == '3':
                    start_date = input("Start date (YYYY-MM-DD): ").strip()
                    end_date = input("End date (YYYY-MM-DD): ").strip()
                else:
                    start_date = '2020-01-01'
                    end_date = '2022-01-01'
                
                # Run inspection
                long_signals = inspect_long_signals_bull_cycle(
                    df, short_model, long_model, 
                    start_date=start_date, 
                    end_date=end_date
                )
                
                # Save results
                if long_signals:
                    df_results = pd.DataFrame(long_signals)
                    df_results.to_csv('validation/long_signals_bull_cycle.csv', index=False)
                    print(f"\n✅ Saved: validation/long_signals_bull_cycle.csv")
            else:
                print("❌ Models not found. Run option 1 first.")
    
    elif choice == '7':  # NEW OPTION
        trades = run_90_day_shadow_trading()
        
        print("\n📋 Shadow trading complete!")
        print("\n📊 Output files:")
        print("   shadow_trading/shadow_trades_90d.csv - Trade log")
        print("   shadow_trading/shadow_analysis.md - Detailed analysis")
        
        if trades:
            print("\n🎯 Next steps:")
            print("   1. Review shadow_analysis.md")
            print("   2. Answer performance questions")
            print("   3. If ≥3/4 positive → system is production-grade")
            print("   4. Consider MAE-aware sizing if needed")
    
    elif choice == '8':
        test_trades = test_shadow_trading_basic()

        print("   8. [DEBUG] Test shadow trading system")

    else:
        print("❌ Invalid option")
        
    print("\n" + "="*70)
    print("SYSTEM STATUS CHECK")
    print("="*70)
    print("✅ Signal logic: Frozen")
    print("✅ Regimes: Stable")
    print("✅ Funding: Optional modifier")
    print("✅ Output contract: Consistent")
    print(f"📊 Next: {'90-day shadow trading' if choice != '7' else 'Review results'}")
    print(f"{'='*70}")

✅ Modular shadow trading system imported successfully

ETH WHALE ALPHA PIPELINE - UNIFIED SIGNAL CONTRACT
PHASE: 90-DAY SHADOW TRADING

FILE SYSTEM CHECK
✅ shadow_trading: Writable
✅ validation: Writable
✅ backtest: Writable
✅ models: Writable
✅ data: Writable
✅ logs/daily: Writable

📁 Current directory: c:\Users\HP\Documents\Whale-Movement-Based-Price-Direction-Generator-V2\WhalesIntent\Intent
📁 Directory contents: 11 items
📁 shadow_trading contents: ['shadow_analysis.py', 'shadow_trader.py', '__init__.py', '__pycache__']

📋 Available Options:
   1. Run unified pipeline (train models + generate signal)
   2. 60-day paper trade test (unified logic)
   3. Manual signal review (30 days)
   4. Inspect signals (60 days)
   5. Load and check data only
   6. Extended LONG inspection (2020-2021 bull cycle)
   7. [NEW] 90-day shadow trading with MAE/MFE
   8. [DEBUG] Test shadow trading system

🧪 Testing Shadow Trading System...
✅ ShadowTrader initialized
   Output directory: shadow_trading
✅ 

In [2]:
import sys
sys.path.append('.')
from shadow_trading.shadow_trader import ShadowTrader
   
print(ShadowTrader)

<class 'shadow_trading.shadow_trader.ShadowTrader'>
